# 04 — Estratégias de carteira e backtest (100 experiências)

Neste notebook comparamos as estratégias da parte 2 do enunciado:

| | Estratégia | Ideia |
|---|---|---|
| a) | **Equally weighted (EW)** | mesmo peso em todas as ações (benchmark principal) |
| b) | **Markowitz média-variância** | maximizar retorno − (λ/2)·variância, com aversão ao risco λ = 3 |
| c) | **Global minimum variance (GMV)** | a carteira com a menor volatilidade possível |
| d) | **Max Sharpe** | a carteira com o maior rácio de Sharpe esperado |
| e) | **Inverse volatility** e **Risk parity (ERC)** | pesos inversos à volatilidade / contribuições de risco iguais |
| f) | **Most diversified** e **Max decorrelation** | maximizar o rácio de diversificação / minimizar a correlação média |

São 8 estratégias. O **SPY** (ETF investível do S&P 500) entra como referência de mercado.

### Convenções (iguais para todas as estratégias)

| Item | Escolha |
|---|---|
| Universo | 23 ações do S&P 500 escolhidas no notebook `01` |
| Dados | preços de fecho ajustados (Yahoo Finance), diários, 2010-01-04 a 2026-09-18 |
| Retornos | **simples** para as carteiras (os log-retornos só foram usados nos factos estilizados) |
| Experiências | **100**, semente aleatória **2026** |
| Cada experiência | um período **contíguo de 3 anos** (756 dias úteis) + **12 ações** sorteadas entre as 23 |
| Divisão | anos 1–2 = estimação; **ano 3 = avaliação fora da amostra** (252 dias) |
| Janela de estimação | móvel, **1 ano** (252 dias), só com dados anteriores à data de rebalanceamento |
| Rebalanceamento | **semestral** (126 dias úteis); entre datas os pesos **derivam** com os preços |
| Restrições | long-only (pesos ≥ 0), totalmente investido (soma = 1), sem alavancagem |
| Taxa sem risco | *yield* da Treasury a **3 anos** (FRED, série `DGS3`); diária = taxa anual / 252 |
| Custos de transação | **10 bps** sobre o valor transacionado (Σ\|Δw\|); robustez com 0, 25 e 50 bps |
| Benchmarks | EW (principal) e SPY buy & hold |

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from IPython.display import display

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 250)

os.makedirs('../graficos', exist_ok=True)
os.makedirs('../resultados', exist_ok=True)

## 1. Parâmetros do backtest

Todos os parâmetros estão aqui em cima, para ser fácil mudar e voltar a correr.

In [ ]:
SEMENTE = 2026              # semente aleatória -> garante que as 100 experiências são sempre as mesmas
N_EXPERIENCIAS = 100        # nº de datasets de 3 anos
N_ATIVOS = 12               # nº de ações sorteadas em cada experiência (de 23)

DIAS_ANO = 252                      # dias úteis num ano
DIAS_DATASET = 3 * DIAS_ANO         # 756 dias = 3 anos
INICIO_AVALIACAO = 2 * DIAS_ANO     # dia 504: os 2 primeiros anos são para estimação
JANELA_ESTIMACAO = DIAS_ANO         # janela móvel de 1 ano para estimar médias e covariâncias
INTERVALO_REBALANCE = 126           # semestral (126 dias úteis = 6 meses)

CUSTO = 0.0010              # 10 bps (0.10%) por cada unidade transacionada
AVERSAO_RISCO = 3           # lambda do Markowitz

## 2. Dados: preços, retornos simples e taxa sem risco

- Usamos **retornos simples** $R_t = P_t / P_{t-1} - 1$ porque o retorno de uma carteira é a média ponderada dos retornos simples dos ativos
  (com log-retornos isso não é verdade).
- Os preços ajustados já incluem dividendos e *splits*. Todas as ações estão em USD e cotadas na bolsa dos EUA, por isso não há conversão cambial
  e o calendário é o mesmo (usámos o calendário do SPY no notebook `01`).
- **SPY vs ^GSPC:** o ^GSPC é o índice S&P 500 — não é investível e o preço não inclui dividendos. O SPY é um ETF que se pode comprar
  (tem custos de gestão de ~0.09%/ano) e o preço ajustado inclui os dividendos. Por isso o benchmark de mercado no backtest é o **SPY**.

In [ ]:
precos = pd.read_csv('data/precos_limpos.csv', index_col=0, parse_dates=True)

# As 23 ações = todas as colunas menos o SPY e o índice ^GSPC
acoes = [coluna for coluna in precos.columns if coluna not in ['SPY', '^GSPC']]
print('Nº de ações no universo:', len(acoes))
print(acoes)

# Retornos simples diários (a primeira linha fica NaN porque não há preço anterior -> retiramos)
retornos = precos.pct_change().iloc[1:]

print('\nPeríodo:', retornos.index.min().date(), 'a', retornos.index.max().date(), '|', len(retornos), 'dias úteis')
print('Valores em falta por ação (só a TSLA, que entrou em bolsa em junho de 2010):')
print(retornos[acoes].isna().sum()[retornos[acoes].isna().sum() > 0])

### Resumo do universo (período completo)

Antes de comparar estratégias, vemos as características de cada ação: retorno e volatilidade anualizados, drawdown máximo
e correlação média com as outras ações. Correlações baixas = mais espaço para diversificar.

In [ ]:
# Setor GICS de cada ação (igual ao notebook 01)
setor = {
    'NVDA': 'Information Technology', 'AAPL': 'Information Technology', 'MSFT': 'Information Technology',
    'AVGO': 'Information Technology', 'AMD': 'Information Technology', 'MU': 'Information Technology',
    'BRK-B': 'Financials', 'JPM': 'Financials', 'V': 'Financials',
    'LLY': 'Health Care', 'JNJ': 'Health Care',
    'AMZN': 'Consumer Discretionary', 'TSLA': 'Consumer Discretionary',
    'T': 'Communication Services', 'GOOGL': 'Communication Services',
    'GE': 'Industrials', 'CAT': 'Industrials',
    'WMT': 'Consumer Staples', 'COST': 'Consumer Staples',
    'XOM': 'Energy', 'NEE': 'Utilities', 'WELL': 'Real Estate', 'LIN': 'Materials',
}

correlacoes = retornos[acoes].corr()      # cada par usa os dias em que as duas ações têm dados

linhas = []
for acao in acoes + ['SPY']:
    r = retornos[acao].dropna()
    riqueza = (1 + r).cumprod()
    linhas.append({
        'Ticker': acao,
        'Sector': setor.get(acao, 'ETF (benchmark)'),
        'First date': r.index.min().date(),
        'Ann. return (CAGR)': riqueza.iloc[-1] ** (252 / len(r)) - 1,
        'Ann. volatility': r.std() * np.sqrt(252),
        'Max drawdown': (riqueza / riqueza.cummax() - 1).min(),
        'Avg. correlation with the other stocks': (correlacoes.loc[acao].sum() - 1) / (len(acoes) - 1) if acao in acoes else np.nan,
    })

resumo_universo = pd.DataFrame(linhas).set_index('Ticker')
display(resumo_universo.round(3))

In [ ]:
# Mapa de calor das correlações dos retornos diários
fig, ax = plt.subplots(figsize=(11, 9))
imagem = ax.imshow(correlacoes.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(acoes)))
ax.set_xticklabels(acoes, rotation=90)
ax.set_yticks(range(len(acoes)))
ax.set_yticklabels(acoes)
for i in range(len(acoes)):
    for j in range(len(acoes)):
        ax.text(j, i, f'{correlacoes.values[i, j]:.1f}', ha='center', va='center', fontsize=7)
plt.colorbar(imagem, label='Correlation')
ax.set_title('Correlation of daily returns, 23 stocks (2010-2026)')
plt.tight_layout()
plt.savefig('../graficos/04_correlacoes_universo.png', dpi=150)
plt.show()

print(f'Correlação média entre pares de ações: {(correlacoes.values.sum() - len(acoes)) / (len(acoes) * (len(acoes) - 1)):.2f}')

### Taxa sem risco: *yield* da Treasury a 3 anos (FRED `DGS3`)

O código tenta ler o ficheiro `data/taxa_sem_risco_DGS3.csv`. Se ainda não existir, descarrega a série do FRED e guarda-a
(data da extração = data em que este código for corrido; fica registada no ficheiro `data/taxa_sem_risco_DGS3_info.txt`;
se o download falhar e descarregarem o CSV à mão, anotem a data do download).

- A série vem em % ao ano (ex.: 4.25 = 4.25%) → dividimos por 100.
- Nos dias em que a bolsa abre mas o mercado de obrigações fecha (ex.: *Columbus Day*), usamos o último valor conhecido.
- Taxa diária = taxa anual / 252.

In [ ]:
FICHEIRO_RF = 'data/taxa_sem_risco_DGS3.csv'
URL_FRED = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS3'

if os.path.exists(FICHEIRO_RF):
    print('A ler a taxa sem risco do ficheiro local:', FICHEIRO_RF)
    taxa = pd.read_csv(FICHEIRO_RF, index_col=0, parse_dates=True, na_values='.')   # o FRED usa '.' nos dias sem valor
else:
    print('A descarregar a série DGS3 do FRED...')
    try:
        taxa = pd.read_csv(URL_FRED, index_col=0, parse_dates=True, na_values='.')
    except Exception as erro:
        # Acontece por exemplo em redes com proxy/firewall (universidade, empresa)
        print('Não foi possível descarregar do FRED:', erro)
        print('SOLUÇÃO: abra este link no browser (descarrega um CSV):', URL_FRED)
        print('e guarde o ficheiro como:', os.path.abspath(FICHEIRO_RF))
        print('Depois corra esta célula outra vez.')
        print('Anote a data de hoje: é a data de extração da taxa sem risco (o relatório tem de indicar a fonte e a data).')
        raise
    taxa.to_csv(FICHEIRO_RF)
    with open('data/taxa_sem_risco_DGS3_info.txt', 'w') as f:
        f.write(f'Fonte: {URL_FRED}\nExtraído em: {pd.Timestamp.today().date()}\n')

# Ficar só com a coluna dos valores, passar de % para decimal e retirar dias sem valor
taxa = taxa.iloc[:, 0] / 100
taxa = taxa.dropna()

# Alinhar com os dias de negociação das ações (method='ffill' = usar o último valor conhecido)
rf_anual = taxa.reindex(retornos.index, method='ffill')
rf_diaria = rf_anual / DIAS_ANO

# Verificação: a taxa tem de existir em TODOS os dias (senão o Sharpe e o Max Sharpe ficam errados)
if rf_anual.isna().sum() > 0:
    raise ValueError('A taxa sem risco não cobre todo o período das ações (começa em ' + str(taxa.index.min().date()) +
                     '). Apague ' + FICHEIRO_RF + ' e descarregue a série COMPLETA do FRED.')

print('Período da taxa:', taxa.index.min().date(), 'a', taxa.index.max().date())
print(f'Taxa média no período: {rf_anual.mean():.2%} | mínimo: {rf_anual.min():.2%} | máximo: {rf_anual.max():.2%}')

plt.figure(figsize=(10, 3.5))
plt.plot(rf_anual.index, rf_anual * 100, color='darkgreen')
plt.title('Risk-free rate: 3-year US Treasury yield (FRED DGS3)')
plt.ylabel('% per year')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('../graficos/04_taxa_sem_risco.png', dpi=150)
plt.show()

## 3. As 8 estratégias

Notação: $w$ = vetor de pesos, $\mu$ = retornos esperados anualizados, $\Sigma$ = matriz de covariância anualizada,
$\sigma_i$ = volatilidade do ativo *i*, $C$ = matriz de correlações, $r_f$ = taxa sem risco.
**Restrições comuns:** $w_i \ge 0$ e $\sum_i w_i = 1$.

| Estratégia | Objetivo | Usa $\mu$? | Usa $\Sigma$? |
|---|---|---|---|
| EW | $w_i = 1/N$ | não | não |
| Markowitz (λ = 3) | $\max_w \; w^\top\mu - \frac{\lambda}{2}\, w^\top\Sigma w$ | sim | sim |
| GMV | $\min_w \; w^\top\Sigma w$ | não | sim |
| Max Sharpe | $\max_w \; (w^\top\mu - r_f)\,/\,\sqrt{w^\top\Sigma w}$ | sim | sim |
| Inverse vol | $w_i \propto 1/\sigma_i$ | não | só a diagonal |
| Risk parity (ERC) | contribuições de risco iguais: $w_i(\Sigma w)_i / w^\top\Sigma w = 1/N$ | não | sim |
| Most diversified | $\max_w \; DR = w^\top\sigma \,/\, \sqrt{w^\top\Sigma w}$ | não | sim |
| Max decorrelation | $\min_w \; w^\top C\, w$ | não | só as correlações |

**Porque é que o Markowitz é diferente do GMV e do Max Sharpe?** Com λ = 3 o investidor aceita algum risco em troca de retorno:
quando λ → ∞ o Markowitz aproxima-se do GMV; com λ pequeno vai para o ativo com maior retorno esperado.
O Max Sharpe é o ponto da fronteira tangente à taxa sem risco, que em geral não coincide com λ = 3 (ver o gráfico da secção 4).

**Inverse vol vs ERC:** as duas dão mais peso aos ativos menos voláteis, mas o inverse vol **ignora as correlações**.
Só são iguais se todas as correlações forem iguais. Com correlações diferentes, o ERC dá menos peso aos ativos muito correlacionados com os outros.

Os otimizadores usam `scipy.optimize.minimize` com o método **SLSQP** (aceita limites e restrições de igualdade).

In [ ]:
def estimar_parametros(amostra):
    # Média (vetor) e covariância (matriz) ANUALIZADAS a partir de retornos diários
    mu = amostra.mean().values * DIAS_ANO
    sigma = amostra.cov().values * DIAS_ANO
    return mu, sigma


registo_falhas = []   # aqui guardamos as otimizações que falham (o enunciado pede para não as esconder)


def otimizar(funcao_objetivo, n_ativos, nome, pesos_iniciais=None, peso_minimo=0.0):
    # Minimiza 'funcao_objetivo' com as restrições: peso_minimo <= w_i <= 1 e soma dos pesos = 1
    if pesos_iniciais is None:
        pesos_iniciais = np.ones(n_ativos) / n_ativos          # começar no equally weighted

    limites = [(peso_minimo, 1)] * n_ativos
    restricao_soma = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

    # ftol = precisão pedida ao otimizador. Com valores demasiado pequenos (ex.: 1e-12) o SLSQP às vezes
    # já está no ótimo mas diz que falhou; 1e-10 é preciso e estável.
    resultado = minimize(funcao_objetivo, pesos_iniciais, method='SLSQP',
                         bounds=limites, constraints=[restricao_soma],
                         options={'maxiter': 1000, 'ftol': 1e-10})

    if not resultado.success:
        # Plano B: se falhar, usamos equally weighted e registamos a falha
        registo_falhas.append({'Strategy': nome, 'Message': resultado.message})
        return np.ones(n_ativos) / n_ativos

    pesos = np.clip(resultado.x, 0, None)   # tirar pequenos negativos numéricos (ex.: -1e-15)
    return pesos / pesos.sum()

Cada estratégia é uma função que recebe a **amostra de estimação** (retornos diários da janela móvel) e a **taxa sem risco anual**
e devolve um vetor de pesos. Nota: em Python, `@` é o produto matricial (ex.: `w @ sigma @ w` = $w^\top\Sigma w$ = variância da carteira).

In [ ]:
# a) Equally weighted
def pesos_equal_weight(amostra, rf):
    n = amostra.shape[1]
    return np.ones(n) / n


# b) Markowitz média-variância com aversão ao risco lambda
def pesos_markowitz(amostra, rf, aversao=AVERSAO_RISCO):
    mu, sigma = estimar_parametros(amostra)

    def objetivo(w):
        utilidade = w @ mu - aversao / 2 * (w @ sigma @ w)
        return -utilidade                   # minimizar o negativo = maximizar a utilidade

    return otimizar(objetivo, len(mu), 'Markowitz')


# c) Global minimum variance
def pesos_min_variancia(amostra, rf):
    mu, sigma = estimar_parametros(amostra)

    def objetivo(w):
        return w @ sigma @ w                # variância da carteira

    return otimizar(objetivo, len(mu), 'GMV')


# d) Max Sharpe (carteira tangente)
def pesos_max_sharpe(amostra, rf):
    mu, sigma = estimar_parametros(amostra)

    def objetivo(w):
        sharpe = (w @ mu - rf) / np.sqrt(w @ sigma @ w)
        return -sharpe

    return otimizar(objetivo, len(mu), 'Max Sharpe')

In [ ]:
# e1) Inverse volatility (não precisa de otimizador)
def pesos_inverse_vol(amostra, rf):
    vol = amostra.std().values
    return (1 / vol) / np.sum(1 / vol)


def contribuicoes_risco(w, sigma):
    # Percentagem do risco total da carteira que vem de cada ativo (soma = 1)
    variancia = w @ sigma @ w
    return w * (sigma @ w) / variancia


# e2) Risk parity / Equal Risk Contribution
def pesos_risk_parity(amostra, rf):
    mu, sigma = estimar_parametros(amostra)
    n = len(mu)

    def objetivo(w):
        # soma dos quadrados das diferenças entre a contribuição de cada ativo e 1/N
        return np.sum((contribuicoes_risco(w, sigma) - 1 / n) ** 2)

    # Começamos nos pesos inverse vol (já perto da solução) e obrigamos todos os pesos a serem > 0,
    # porque no ERC todos os ativos têm de contribuir para o risco.
    inicio = pesos_inverse_vol(amostra, rf)
    return otimizar(objetivo, n, 'Risk parity (ERC)', pesos_iniciais=inicio, peso_minimo=1e-6)

In [ ]:
def racio_diversificacao(w, sigma):
    # DR = média ponderada das volatilidades / volatilidade da carteira  (>= 1; maior = mais diversificado)
    vol = np.sqrt(np.diag(sigma))
    return (w @ vol) / np.sqrt(w @ sigma @ w)


# f1) Most diversified portfolio
def pesos_max_diversificacao(amostra, rf):
    mu, sigma = estimar_parametros(amostra)

    def objetivo(w):
        return -racio_diversificacao(w, sigma)

    return otimizar(objetivo, len(mu), 'Most diversified')


# f2) Maximum decorrelation: é a "mínima variância" mas usando a matriz de CORRELAÇÕES
#     (como se todos os ativos tivessem a mesma volatilidade)
def pesos_max_descorrelacao(amostra, rf):
    correlacao = amostra.corr().values

    def objetivo(w):
        return w @ correlacao @ w

    return otimizar(objetivo, correlacao.shape[0], 'Max decorrelation')

In [ ]:
# Dicionário com todas as estratégias: nome -> função
estrategias = {
    'EW': pesos_equal_weight,
    'Markowitz (λ=3)': pesos_markowitz,
    'GMV': pesos_min_variancia,
    'Max Sharpe': pesos_max_sharpe,
    'Inverse vol': pesos_inverse_vol,
    'Risk parity (ERC)': pesos_risk_parity,
    'Most diversified': pesos_max_diversificacao,
    'Max decorrelation': pesos_max_descorrelacao,
}

# Ordem e cores usadas em todos os gráficos
ordem = list(estrategias.keys()) + ['SPY (buy & hold)']
cores = dict(zip(ordem, plt.cm.tab10.colors))

## 4. Ilustração: pesos de cada estratégia com o último ano de dados (23 ações)

Antes do backtest, vemos o que cada estratégia faz com **todas as 23 ações** e o último ano de dados
(como se rebalanceássemos hoje). Isto ajuda a perceber as diferenças entre elas.

In [ ]:
amostra_ilustracao = retornos[acoes].iloc[-JANELA_ESTIMACAO:]      # último ano (252 dias)
rf_ilustracao = rf_anual.iloc[-1]
print('Amostra:', amostra_ilustracao.index.min().date(), 'a', amostra_ilustracao.index.max().date(),
      f'| taxa sem risco = {rf_ilustracao:.2%}')

pesos_ilustracao = pd.DataFrame(index=acoes)
for nome, funcao in estrategias.items():
    pesos_ilustracao[nome] = funcao(amostra_ilustracao, rf_ilustracao)

display((100 * pesos_ilustracao).round(1))

In [ ]:
# Características ex-ante (dentro da amostra) de cada carteira
mu_il, sigma_il = estimar_parametros(amostra_ilustracao)

linhas = []
for nome in estrategias.keys():
    w = pesos_ilustracao[nome].values
    retorno = w @ mu_il
    vol = np.sqrt(w @ sigma_il @ w)
    linhas.append({
        'Strategy': nome,
        'Exp. return': retorno,
        'Volatility': vol,
        'Sharpe (ex-ante)': (retorno - rf_ilustracao) / vol,
        'Effective N (1/Σw²)': 1 / np.sum(w ** 2),
        'Max weight': w.max(),
        'N assets with w > 1%': int(np.sum(w > 0.01)),
        'Diversification ratio': racio_diversificacao(w, sigma_il),
    })

caracteristicas = pd.DataFrame(linhas).set_index('Strategy')
display(caracteristicas.round(3))

In [ ]:
# Gráfico dos pesos (uma barra horizontal por estratégia, empilhada por ação)
fig, ax = plt.subplots(figsize=(13, 6))
esquerda = np.zeros(len(estrategias))
cores_acoes = plt.cm.tab20(np.linspace(0, 1, len(acoes)))

for i, acao in enumerate(acoes):
    valores = pesos_ilustracao.loc[acao].values
    ax.barh(list(estrategias.keys()), valores, left=esquerda, color=cores_acoes[i], label=acao, edgecolor='white', linewidth=0.5)
    esquerda = esquerda + valores

ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel('Weight')
ax.set_title('Portfolio weights with the last year of data (23 stocks)')
ax.legend(ncol=2, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('../graficos/04_pesos_ilustracao.png', dpi=150)
plt.show()

### Inverse vol vs Risk parity (ERC): contribuições de risco

No inverse vol os **pesos** são proporcionais a $1/\sigma_i$, mas as **contribuições para o risco** não ficam iguais,
porque os ativos mais correlacionados com o resto da carteira contribuem mais. O ERC corrige isso.

In [ ]:
rc_iv = contribuicoes_risco(pesos_ilustracao['Inverse vol'].values, sigma_il)
rc_erc = contribuicoes_risco(pesos_ilustracao['Risk parity (ERC)'].values, sigma_il)

tab_rc = pd.DataFrame({
    'Weight IV': pesos_ilustracao['Inverse vol'],
    'Risk contrib. IV': rc_iv,
    'Weight ERC': pesos_ilustracao['Risk parity (ERC)'],
    'Risk contrib. ERC': rc_erc,
    'Avg. correlation with others': (amostra_ilustracao.corr().sum() - 1) / (len(acoes) - 1),
})
display((tab_rc * 100).round(2).sort_values('Risk contrib. IV', ascending=False))
print(f'Desvio-padrão das contribuições de risco -> IV: {rc_iv.std():.4f} | ERC: {rc_erc.std():.6f}')

x = np.arange(len(acoes))
plt.figure(figsize=(13, 4))
plt.bar(x - 0.2, rc_iv * 100, width=0.4, label='Inverse vol', color=cores['Inverse vol'])
plt.bar(x + 0.2, rc_erc * 100, width=0.4, label='Risk parity (ERC)', color=cores['Risk parity (ERC)'])
plt.axhline(100 / len(acoes), color='black', linestyle='--', linewidth=1, label='1/N')
plt.xticks(x, acoes, rotation=60)
plt.ylabel('% of portfolio risk')
plt.title('Risk contributions: inverse volatility vs equal risk contribution')
plt.legend()
plt.tight_layout()
plt.savefig('../graficos/04_contribuicoes_risco_iv_vs_erc.png', dpi=150)
plt.show()

### Fronteira eficiente (dentro da amostra)

Calculamos a fronteira long-only: para cada retorno-alvo, a carteira de menor variância. Marcamos as 8 estratégias
e o Markowitz com λ = 1, 3 e 5 para mostrar que é diferente do GMV e do Max Sharpe.

In [ ]:
n = len(acoes)
retorno_gmv = pesos_ilustracao['GMV'].values @ mu_il
alvos = np.linspace(retorno_gmv, mu_il.max(), 40)

fronteira_vol = []
for alvo in alvos:
    restricoes = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
                  {'type': 'eq', 'fun': lambda w: w @ mu_il - alvo}]     # retorno da carteira = alvo
    res = minimize(lambda w: w @ sigma_il @ w, np.ones(n) / n, method='SLSQP',
                   bounds=[(0, 1)] * n, constraints=restricoes, options={'maxiter': 1000})
    fronteira_vol.append(np.sqrt(res.fun))

plt.figure(figsize=(10, 6))
plt.plot(fronteira_vol, alvos, color='black', linewidth=1.5, label='Efficient frontier (long-only)')
plt.scatter(np.sqrt(np.diag(sigma_il)), mu_il, color='lightgrey', edgecolor='grey', s=30, label='Individual stocks')

for nome in estrategias.keys():
    w = pesos_ilustracao[nome].values
    plt.scatter(np.sqrt(w @ sigma_il @ w), w @ mu_il, color=cores[nome], s=90, edgecolor='black', zorder=3, label=nome)

for aversao in [1, 5]:
    w = pesos_markowitz(amostra_ilustracao, rf_ilustracao, aversao=aversao)
    plt.scatter(np.sqrt(w @ sigma_il @ w), w @ mu_il, color=cores['Markowitz (λ=3)'], marker='^', s=70, zorder=3)
    plt.annotate(f'λ={aversao}', (np.sqrt(w @ sigma_il @ w), w @ mu_il), textcoords='offset points', xytext=(6, -10))

plt.xlabel('Volatility (annualised)')
plt.ylabel('Expected return (annualised)')
plt.title('In-sample efficient frontier and strategy portfolios (last 12 months)')
plt.legend(fontsize=8, loc='best')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('../graficos/04_fronteira_eficiente.png', dpi=150)
plt.show()

**Atenção:** esta ilustração é **dentro da amostra** — usa médias históricas de 1 ano como se fossem o futuro.
As estratégias que dependem de $\mu$ (Markowitz e Max Sharpe) concentram-se nas ações que mais subiram no último ano.
É exatamente por isso que precisamos do backtest fora da amostra: ver se essa vantagem aparente se mantém no ano seguinte.

## 5. Desenho das 100 experiências

Para cada experiência sorteamos:
1. **Um período contíguo de 3 anos** (756 dias úteis) dentro de 2010–2026 — a ordem cronológica é mantida;
2. **12 ações** entre as que têm dados completos nesse período (*eligibility*: a TSLA só é elegível em períodos que começam depois de 29/06/2010).

Usamos a **mesma** amostra (período + ações) para todas as estratégias, e a semente fica registada → o resultado é reproduzível.

In [ ]:
# np.random.seed: com a mesma semente, o NumPy garante a mesma sequência de números em qualquer versão/computador
np.random.seed(SEMENTE)
datas = retornos.index
n_dias = len(datas)

linhas = []
for e in range(1, N_EXPERIENCIAS + 1):
    # 1) sortear o 1.º dia do período de 3 anos (tem de caber inteiro nos dados)
    inicio = int(np.random.randint(0, n_dias - DIAS_DATASET + 1))
    fim = inicio + DIAS_DATASET                        # exclusivo

    # 2) ações elegíveis = sem valores em falta no período
    bloco = retornos[acoes].iloc[inicio:fim]
    elegiveis = [acao for acao in acoes if bloco[acao].notna().all()]

    # 3) sortear 12 ações (sem repetição)
    escolhidas = sorted(str(a) for a in np.random.choice(elegiveis, size=N_ATIVOS, replace=False))

    linhas.append({
        'Experiment': e,
        'start_index': inicio,
        'Estimation start': datas[inicio].date(),
        'Evaluation start': datas[inicio + INICIO_AVALIACAO].date(),
        'Evaluation end': datas[fim - 1].date(),
        'N eligible': len(elegiveis),
        'Assets': escolhidas,
    })

desenho = pd.DataFrame(linhas).set_index('Experiment')
display(desenho.head(10))
print('Experiências em que a TSLA não era elegível:', (desenho['N eligible'] < len(acoes)).sum())

In [ ]:
# Gráfico 1: períodos sorteados (cinzento = estimação, azul = avaliação)
desenho_ordenado = desenho.sort_values('start_index')

fig, axes = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={'width_ratios': [2, 1]})
for y, (e, linha) in enumerate(desenho_ordenado.iterrows()):
    axes[0].plot([linha['Estimation start'], linha['Evaluation start']], [y, y], color='lightgrey', linewidth=2)
    axes[0].plot([linha['Evaluation start'], linha['Evaluation end']], [y, y], color='steelblue', linewidth=2)
axes[0].set_title('100 sampled 3-year periods (grey = estimation, blue = out-of-sample year)')
axes[0].set_ylabel('Experiment (sorted by start date)')

# Gráfico 2: quantas vezes cada ação foi sorteada
contagem = pd.Series(0, index=acoes)
for lista in desenho['Assets']:
    for acao in lista:
        contagem[acao] += 1
contagem.sort_values().plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Times each stock was selected')
axes[1].set_xlabel('Number of experiments')

plt.tight_layout()
plt.savefig('../graficos/04_desenho_experiencias.png', dpi=150)
plt.show()

**Nota importante:** os períodos sorteados **sobrepõem-se** (há apenas ~16 anos de dados, por isso há no máximo ~14 anos de avaliação
diferentes). As 100 experiências **não são 100 testes independentes** — servem para ver a **distribuição** dos resultados e se as conclusões
dependem do período ou das ações sorteadas, e não para fazer testes de significância como se fossem independentes.

## 6. Simulação de uma estratégia numa experiência

Para cada experiência e estratégia:

1. **Datas de rebalanceamento:** de 126 em 126 dias. A primeira é **um semestre antes** do início da avaliação (aquecimento):
   assim, no 1.º dia de avaliação a carteira já existe e o turnover desse dia é realista (não é uma compra a partir de dinheiro).
   Os retornos do aquecimento **não** entram nas métricas.
2. Em cada rebalanceamento, os pesos são calculados **só com os 252 dias anteriores** (sem olhar para o futuro).
   As transações são feitas ao fecho do dia anterior, ou seja, os novos pesos aplicam-se a partir do dia *t*
   (convenção *market-on-close*: negociamos ao preço de fecho do dia em que conhecemos a última observação).
3. **Turnover** = $\sum_i |w_i^{alvo} - w_i^{antes}|$ (compras + vendas, em % da carteira).
4. Entre rebalanceamentos os pesos **derivam**: $w_{i,t+1} = w_{i,t}(1+R_{i,t}) / (1+R_{p,t})$.
5. **Retorno líquido** no dia do rebalanceamento = retorno bruto − custo × turnover.

In [ ]:
def simular(retornos_bloco, rf_bloco, funcao_pesos, intervalo=INTERVALO_REBALANCE, janela=JANELA_ESTIMACAO):
    # retornos_bloco: DataFrame com 756 dias x 12 ações (retornos simples)
    # rf_bloco: Series com 756 dias de taxa sem risco ANUAL
    # Devolve um dicionário com os retornos brutos e o turnover de cada dia do ano de avaliação,
    # e os pesos-alvo / rácio de diversificação de cada rebalanceamento do ano de avaliação.
    R = retornos_bloco.values                 # numpy é mais rápido dentro de um ciclo
    n_ativos = R.shape[1]

    # Dias de rebalanceamento (o 1.º é um intervalo antes do início da avaliação = aquecimento)
    dias_rebalance = list(range(INICIO_AVALIACAO - intervalo, DIAS_DATASET, intervalo))
    if dias_rebalance[0] - janela < 0:
        raise ValueError('A janela de estimação não cabe antes do 1.º rebalanceamento')

    pesos = np.zeros(n_ativos)              # começamos em dinheiro
    retornos_brutos = []
    turnover_diario = []
    pesos_alvo = []
    racios_div = []

    for t in range(dias_rebalance[0], DIAS_DATASET):
        turnover = 0.0

        if t in dias_rebalance:
            # 1) estimar só com dados PASSADOS (dias t-janela até t-1)
            amostra = retornos_bloco.iloc[t - janela:t]
            rf_conhecida = rf_bloco.iloc[t - 1]
            n_falhas_antes = len(registo_falhas)
            alvo = funcao_pesos(amostra, rf_conhecida)
            for falha in registo_falhas[n_falhas_antes:]:          # se a otimização falhou, guardar a data
                falha['Rebalance date'] = retornos_bloco.index[t].date()

            # 2) quanto é preciso transacionar para passar dos pesos atuais para o alvo
            turnover = np.sum(np.abs(alvo - pesos))
            pesos = alvo

            if t >= INICIO_AVALIACAO:
                pesos_alvo.append(alvo)
                racios_div.append(racio_diversificacao(alvo, amostra.cov().values))

        # 3) retorno da carteira no dia t
        r_carteira = np.sum(pesos * R[t])

        # 4) os pesos derivam com os preços até ao próximo rebalanceamento
        pesos = pesos * (1 + R[t]) / (1 + r_carteira)

        # 5) só guardamos o ano de avaliação
        if t >= INICIO_AVALIACAO:
            retornos_brutos.append(r_carteira)
            turnover_diario.append(turnover)

    datas_avaliacao = retornos_bloco.index[INICIO_AVALIACAO:]
    return {
        'r_bruto': pd.Series(retornos_brutos, index=datas_avaliacao),
        'turnover': pd.Series(turnover_diario, index=datas_avaliacao),
        'pesos': pesos_alvo,
        'div_ratio': racios_div,
    }

## 7. Métricas de desempenho (convenções)

Calculadas sobre os retornos **diários** do ano de avaliação (252 dias):

| Métrica | Fórmula / convenção |
|---|---|
| Retorno anualizado | $\left(\prod_t (1+R_t)\right)^{252/n} - 1$ (geométrico) |
| Volatilidade anualizada | $\text{desvio-padrão}(R_t)\times\sqrt{252}$ |
| Sharpe | $\overline{(R_t - r_{f,t})} \,/\, \text{dp}(R_t - r_{f,t}) \times \sqrt{252}$ |
| Sortino | $\overline{(R_t - r_{f,t})} \,/\, DD \times \sqrt{252}$, com $DD = \sqrt{\text{média}(\min(R_t - r_{f,t}, 0)^2)}$ |
| Drawdown máximo (MDD) | maior queda desde um máximo anterior da riqueza (começa em 1) |
| Calmar | retorno anualizado / \|MDD\| |
| Sterling | retorno anualizado / (\|MDD\| + 10%) — versão original; com 1 ano de avaliação o "MDD médio anual" é o MDD desse ano |
| Turnover anual | soma do turnover dos rebalanceamentos do ano de avaliação |
| Cost drag | retorno anualizado bruto − retorno anualizado líquido |
| N efetivo | $1/\sum_i w_i^2$ (média dos rebalanceamentos): 12 = pesos iguais, 1 = tudo numa ação |
| Max weight | maior peso-alvo da carteira (média dos rebalanceamentos) |
| Diversification ratio (ex-ante) | $w^\top\sigma / \sqrt{w^\top\Sigma w}$ calculado com a **amostra de estimação** (os 252 dias antes do rebalanceamento), média dos rebalanceamentos. É uma medida *ex-ante* (dentro da amostra), não um resultado fora da amostra — o *Most diversified* é o maior por construção |

Os rácios usam a **taxa sem risco de cada dia** (Sharpe, Sortino). Calmar e Sterling usam o retorno anualizado (sem subtrair a taxa sem risco), como nas definições originais.

In [ ]:
def calcular_metricas(r, rf_d):
    # r: retornos diários da carteira | rf_d: taxa sem risco diária nas mesmas datas
    excesso = r - rf_d
    n = len(r)

    riqueza = (1 + r).cumprod()
    retorno_anual = riqueza.iloc[-1] ** (DIAS_ANO / n) - 1
    vol_anual = r.std() * np.sqrt(DIAS_ANO)

    sharpe = excesso.mean() / excesso.std() * np.sqrt(DIAS_ANO)

    desvio_negativo = np.sqrt(np.mean(np.minimum(excesso, 0) ** 2))
    sortino = excesso.mean() / desvio_negativo * np.sqrt(DIAS_ANO)

    # Drawdown: incluímos o valor inicial 1 para contar uma queda logo no 1.º dia
    riqueza_com_inicio = np.concatenate([[1.0], riqueza.values])
    maximo_ate_agora = np.maximum.accumulate(riqueza_com_inicio)
    drawdown = riqueza_com_inicio / maximo_ate_agora - 1
    mdd = drawdown.min()                                    # número negativo

    calmar = retorno_anual / abs(mdd) if mdd < 0 else np.nan
    sterling = retorno_anual / (abs(mdd) + 0.10)

    return {
        'Ann. return': retorno_anual,
        'Ann. volatility': vol_anual,
        'Sharpe': sharpe,
        'Sortino': sortino,
        'Max drawdown': mdd,
        'Calmar': calmar,
        'Sterling': sterling,
    }

In [ ]:
def tabela_metricas(simulacoes, custo, tabela_desenho=None):
    # Transforma as simulações numa tabela: uma linha por (experiência, estratégia)
    # tabela_desenho: por defeito usa o desenho principal das experiências
    if tabela_desenho is None:
        tabela_desenho = desenho

    linhas = []
    for (e, nome), sim in simulacoes.items():
        rf_d = rf_diaria.loc[sim['r_bruto'].index]

        r_liquido = sim['r_bruto'] - custo * sim['turnover']
        m_liquido = calcular_metricas(r_liquido, rf_d)
        m_bruto = calcular_metricas(sim['r_bruto'], rf_d)

        linha = {'Experiment': e, 'Strategy': nome}
        linha.update(m_liquido)                            # métricas líquidas de custos
        linha['Ann. return (gross)'] = m_bruto['Ann. return']
        linha['Sharpe (gross)'] = m_bruto['Sharpe']
        linha['Cost drag'] = m_bruto['Ann. return'] - m_liquido['Ann. return']
        linha['Turnover (annual)'] = sim['turnover'].sum()
        linha['Effective N'] = np.mean([1 / np.sum(w ** 2) for w in sim['pesos']])
        linha['Max weight'] = np.mean([w.max() for w in sim['pesos']])
        linha['Diversification ratio (ex-ante)'] = np.mean(sim['div_ratio'])
        linhas.append(linha)

    # Benchmark de mercado: SPY comprado no início do ano de avaliação e mantido (sem custos de rebalanceamento)
    for e in tabela_desenho.index:
        inicio = tabela_desenho.loc[e, 'start_index']
        datas_avaliacao = retornos.index[inicio + INICIO_AVALIACAO: inicio + DIAS_DATASET]
        m_spy = calcular_metricas(retornos.loc[datas_avaliacao, 'SPY'], rf_diaria.loc[datas_avaliacao])
        linha = {'Experiment': e, 'Strategy': 'SPY (buy & hold)'}
        linha.update(m_spy)
        linha['Ann. return (gross)'] = m_spy['Ann. return']
        linha['Sharpe (gross)'] = m_spy['Sharpe']
        linha['Cost drag'] = 0.0
        linha['Turnover (annual)'] = 0.0
        linhas.append(linha)

    return pd.DataFrame(linhas)


def correr_backtest(dicionario_estrategias, intervalo=INTERVALO_REBALANCE, tabela_desenho=None, nome_corrida='Main'):
    # Corre todas as estratégias em todas as experiências. Devolve um dicionário (experiência, estratégia) -> simulação
    # nome_corrida: identifica esta corrida no registo de falhas de otimização
    if tabela_desenho is None:
        tabela_desenho = desenho

    simulacoes = {}
    for e in tabela_desenho.index:
        inicio = tabela_desenho.loc[e, 'start_index']
        ativos_e = tabela_desenho.loc[e, 'Assets']
        bloco = retornos[ativos_e].iloc[inicio:inicio + DIAS_DATASET]
        rf_bloco = rf_anual.iloc[inicio:inicio + DIAS_DATASET]

        for nome, funcao in dicionario_estrategias.items():
            n_falhas_antes = len(registo_falhas)
            simulacoes[(e, nome)] = simular(bloco, rf_bloco, funcao, intervalo=intervalo)
            for falha in registo_falhas[n_falhas_antes:]:       # identificar as falhas desta simulação
                falha['Run'] = nome_corrida
                falha['Experiment'] = e
                falha['Strategy'] = nome                          # nome completo, ex.: 'Markowitz (λ=1)'
    return simulacoes


def mostrar_falhas(nome_corrida):
    # Mostra as otimizações que falharam numa corrida (nesses rebalanceamentos usámos pesos iguais)
    falhas = [f for f in registo_falhas if f.get('Run') == nome_corrida]
    print(f'Otimizações que falharam ({nome_corrida}): {len(falhas)}')
    if len(falhas) > 0:
        display(pd.DataFrame(falhas))

## 8. Correr o backtest principal (rebalanceamento semestral, 10 bps)

In [ ]:
import time
inicio_relogio = time.time()

simulacoes = correr_backtest(estrategias, nome_corrida='Main (semi-annual)')
resultados = tabela_metricas(simulacoes, CUSTO)

print(f'Backtest concluído em {time.time() - inicio_relogio:.0f} segundos')
print('Nº de simulações:', len(simulacoes), '| linhas na tabela (inclui SPY):', len(resultados))
mostrar_falhas('Main (semi-annual)')

In [ ]:
display(resultados.head(10).round(4))

## 9. Resultados nas 100 experiências

### 9.1 Mediana de cada métrica (líquida de custos, 10 bps)

In [ ]:
colunas_resumo = ['Ann. return', 'Ann. volatility', 'Sharpe', 'Sortino', 'Max drawdown', 'Calmar', 'Sterling',
                  'Ann. return (gross)', 'Sharpe (gross)', 'Cost drag', 'Turnover (annual)',
                  'Effective N', 'Max weight', 'Diversification ratio (ex-ante)']

mediana = resultados.groupby('Strategy')[colunas_resumo].median().loc[ordem]
display(mediana.round(3))

### 9.2 Dispersão entre experiências e % de experiências em que cada estratégia bate o EW

In [ ]:
# Dispersão: percentis 25 e 75 do Sharpe, do retorno e do Sterling (líquidos)
dispersao = pd.DataFrame({
    'Sharpe P25': resultados.groupby('Strategy')['Sharpe'].quantile(0.25),
    'Sharpe median': resultados.groupby('Strategy')['Sharpe'].median(),
    'Sharpe P75': resultados.groupby('Strategy')['Sharpe'].quantile(0.75),
    'Return P25': resultados.groupby('Strategy')['Ann. return'].quantile(0.25),
    'Return median': resultados.groupby('Strategy')['Ann. return'].median(),
    'Return P75': resultados.groupby('Strategy')['Ann. return'].quantile(0.75),
    'Return std': resultados.groupby('Strategy')['Ann. return'].std(),
    'Sterling P25': resultados.groupby('Strategy')['Sterling'].quantile(0.25),
    'Sterling median': resultados.groupby('Strategy')['Sterling'].median(),
    'Sterling P75': resultados.groupby('Strategy')['Sterling'].quantile(0.75),
}).loc[ordem]
display(dispersao.round(3))


def percentagem_bate_ew(tabela):
    # % de experiências em que a estratégia é melhor do que o EW na MESMA experiência
    resultado = pd.DataFrame(index=ordem)
    for metrica, maior_e_melhor in [('Sharpe', True), ('Ann. return', True), ('Max drawdown', True),
                                     ('Ann. volatility', False), ('Sterling', True)]:
        largo = tabela.pivot(index='Experiment', columns='Strategy', values=metrica)   # experiências x estratégias
        if maior_e_melhor:
            bate = largo.gt(largo['EW'], axis=0)       # gt = greater than
        else:
            bate = largo.lt(largo['EW'], axis=0)       # lt = lower than (menos volatilidade é melhor)
        resultado[f'% beats EW: {metrica}'] = 100 * bate.mean()
    return resultado.drop(index='EW')


bate_ew = percentagem_bate_ew(resultados)
display(bate_ew.round(0))

In [ ]:
# Boxplots das principais métricas nas 100 experiências
metricas_grafico = [('Sharpe', 'Sharpe ratio (net)'), ('Ann. return', 'Annualised return (net)'),
                    ('Ann. volatility', 'Annualised volatility'), ('Max drawdown', 'Maximum drawdown'),
                    ('Sortino', 'Sortino ratio (net)'), ('Sterling', 'Sterling ratio (net)')]

fig, axes = plt.subplots(3, 2, figsize=(16, 15))
for ax, (metrica, titulo) in zip(axes.flatten(), metricas_grafico):
    dados = [resultados.loc[resultados['Strategy'] == nome, metrica].values for nome in ordem]
    caixa = ax.boxplot(dados, patch_artist=True, showfliers=True, medianprops=dict(color='black'))
    for patch, nome in zip(caixa['boxes'], ordem):
        patch.set_facecolor(cores[nome])
        patch.set_alpha(0.7)
    ax.set_xticks(range(1, len(ordem) + 1))
    ax.set_xticklabels(ordem, rotation=35, ha='right')
    ax.axhline(np.median(dados[0]), color='grey', linestyle='--', linewidth=1)   # mediana do EW
    ax.set_title(f'{titulo} - 100 experiments (dashed = EW median)')
    ax.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('../graficos/04_boxplots_metricas.png', dpi=150)
plt.show()

In [ ]:
# Retorno vs risco (medianas) e % de experiências que batem o EW
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for nome in ordem:
    axes[0].scatter(mediana.loc[nome, 'Ann. volatility'], mediana.loc[nome, 'Ann. return'],
                    s=120, color=cores[nome], edgecolor='black', zorder=3)
    axes[0].annotate(nome, (mediana.loc[nome, 'Ann. volatility'], mediana.loc[nome, 'Ann. return']),
                     textcoords='offset points', xytext=(7, 4), fontsize=9)
axes[0].set_xlabel('Median annualised volatility')
axes[0].set_ylabel('Median annualised return (net)')
axes[0].set_title('Median risk and return across 100 experiments')
axes[0].grid(True, linestyle='--', alpha=0.3)

bate_ew[['% beats EW: Sharpe', '% beats EW: Ann. return', '% beats EW: Max drawdown', '% beats EW: Sterling']].plot(
    kind='bar', ax=axes[1], color=['steelblue', 'darkorange', 'seagreen', 'purple'])
axes[1].axhline(50, color='black', linestyle='--', linewidth=1)
axes[1].set_ylabel('% of experiments')
axes[1].set_title('Share of experiments in which each strategy beats EW (net of costs)')
axes[1].set_xticklabels(bate_ew.index, rotation=35, ha='right')
axes[1].legend(['Sharpe', 'Return', 'Drawdown (smaller)', 'Sterling'], fontsize=9)

plt.tight_layout()
plt.savefig('../graficos/04_risco_retorno_e_bate_ew.png', dpi=150)
plt.show()

### 9.3 Turnover, custos e concentração

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
estrategias_sem_spy = list(estrategias.keys())

mediana.loc[estrategias_sem_spy, 'Turnover (annual)'].plot(kind='bar', ax=axes[0], color=[cores[n] for n in estrategias_sem_spy])
axes[0].set_title('Median annual turnover (Σ|Δw|)')

(10000 * mediana.loc[estrategias_sem_spy, 'Cost drag']).plot(kind='bar', ax=axes[1], color=[cores[n] for n in estrategias_sem_spy])
axes[1].set_title('Median cost drag (bps per year, 10 bps cost)')

mediana.loc[estrategias_sem_spy, 'Effective N'].plot(kind='bar', ax=axes[2], color=[cores[n] for n in estrategias_sem_spy])
axes[2].axhline(N_ATIVOS, color='black', linestyle='--', linewidth=1)
axes[2].set_title(f'Median effective number of stocks (max = {N_ATIVOS})')

for ax in axes:
    ax.set_xticklabels(estrategias_sem_spy, rotation=35, ha='right')
    ax.grid(True, linestyle='--', alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../graficos/04_turnover_custos_concentracao.png', dpi=150)
plt.show()

### 9.4 Ranking das estratégias em cada experiência

Em cada experiência ordenamos as estratégias pelo Sharpe líquido (1 = melhor). Um ranking médio baixo e estável é bom sinal.

In [ ]:
sharpe_largo = resultados.pivot(index='Experiment', columns='Strategy', values='Sharpe')[ordem]
rankings = sharpe_largo.rank(axis=1, ascending=False)          # rank dentro de cada linha (experiência)

tab_rankings = pd.DataFrame({
    'Average rank': rankings.mean(),
    'Median rank': rankings.median(),
    '% ranked 1st': 100 * (rankings == 1).mean(),
    '% in top 3': 100 * (rankings <= 3).mean(),
    '% ranked last': 100 * (rankings == len(ordem)).mean(),
}).sort_values('Average rank')
display(tab_rankings.round(1))

### 9.5 Exemplo: evolução da riqueza numa experiência

Escolhemos a primeira experiência cujo ano de avaliação inclui o *crash* de março de 2020 (se existir), para ver o comportamento numa crise.

In [ ]:
data_crise = pd.Timestamp('2020-03-16')
candidatas = [e for e in desenho.index
              if pd.Timestamp(desenho.loc[e, 'Evaluation start']) <= data_crise <= pd.Timestamp(desenho.loc[e, 'Evaluation end'])]
exp_exemplo = candidatas[0] if len(candidatas) > 0 else desenho.index[0]
print('Experiência escolhida:', exp_exemplo)
print('Ações:', desenho.loc[exp_exemplo, 'Assets'])
print('Avaliação:', desenho.loc[exp_exemplo, 'Evaluation start'], 'a', desenho.loc[exp_exemplo, 'Evaluation end'])

plt.figure(figsize=(12, 5))
for nome in estrategias.keys():
    sim = simulacoes[(exp_exemplo, nome)]
    r_liquido = sim['r_bruto'] - CUSTO * sim['turnover']
    plt.plot((1 + r_liquido).cumprod(), label=nome, color=cores[nome], linewidth=1.3)

datas_ex = simulacoes[(exp_exemplo, 'EW')]['r_bruto'].index
plt.plot((1 + retornos.loc[datas_ex, 'SPY']).cumprod(), label='SPY (buy & hold)', color='black', linestyle='--', linewidth=1.3)
plt.axhline(1, color='grey', linewidth=0.8)
plt.title(f'Experiment {exp_exemplo}: growth of 1 USD in the out-of-sample year (net of costs)')
plt.ylabel('Wealth')
plt.legend(fontsize=8, ncol=3)
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('../graficos/04_riqueza_exemplo.png', dpi=150)
plt.show()

## 10. Robustez

### 10.1 Custos de transação mais altos (0, 10, 25 e 50 bps)

Não é preciso voltar a simular: basta recalcular os retornos líquidos com outro custo.

In [ ]:
niveis_custo = [0.0, 0.0010, 0.0025, 0.0050]

sharpe_por_custo = pd.DataFrame(index=ordem)
ranking_por_custo = pd.DataFrame(index=ordem)
for custo in niveis_custo:
    tabela_c = tabela_metricas(simulacoes, custo)
    rotulo = f'{custo * 10000:.0f} bps'
    sharpe_por_custo[rotulo] = tabela_c.groupby('Strategy')['Sharpe'].median()
    ranking_por_custo[rotulo] = sharpe_por_custo[rotulo].rank(ascending=False)

print('Sharpe mediano (líquido) para cada nível de custo')
display(sharpe_por_custo.round(3))
print('Posição no ranking (pelo Sharpe mediano)')
display(ranking_por_custo.astype(int))

plt.figure(figsize=(10, 5))
for nome in ordem:
    plt.plot(niveis_custo, sharpe_por_custo.loc[nome], marker='o', color=cores[nome], label=nome)
plt.xticks(niveis_custo, [f'{c * 10000:.0f}' for c in niveis_custo])
plt.xlabel('Transaction cost (bps per unit traded)')
plt.ylabel('Median Sharpe ratio (net)')
plt.title('Sensitivity of the median Sharpe ratio to transaction costs')
plt.legend(fontsize=8, ncol=2)
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('../graficos/04_sensibilidade_custos.png', dpi=150)
plt.show()

### 10.2 Rebalanceamento trimestral (convenção sugerida no enunciado)

Mesmas experiências, mesmas estratégias, mas a rebalancear de 63 em 63 dias.

In [ ]:
simulacoes_trim = correr_backtest(estrategias, intervalo=63, nome_corrida='Quarterly')
resultados_trim = tabela_metricas(simulacoes_trim, CUSTO)
mostrar_falhas('Quarterly')

comparacao_rebalance = pd.DataFrame({
    'Sharpe (semi-annual)': resultados.groupby('Strategy')['Sharpe'].median(),
    'Sharpe (quarterly)': resultados_trim.groupby('Strategy')['Sharpe'].median(),
    'Return (semi-annual)': resultados.groupby('Strategy')['Ann. return'].median(),
    'Return (quarterly)': resultados_trim.groupby('Strategy')['Ann. return'].median(),
    'Turnover (semi-annual)': resultados.groupby('Strategy')['Turnover (annual)'].median(),
    'Turnover (quarterly)': resultados_trim.groupby('Strategy')['Turnover (annual)'].median(),
}).loc[ordem]
comparacao_rebalance['% beats EW Sharpe (quarterly)'] = percentagem_bate_ew(resultados_trim)['% beats EW: Sharpe']
display(comparacao_rebalance.round(3))

### 10.3 Aversão ao risco do Markowitz (λ = 1, 3, 5)

In [ ]:
estrategias_lambda = {
    'Markowitz (λ=1)': lambda amostra, rf: pesos_markowitz(amostra, rf, aversao=1),
    'Markowitz (λ=5)': lambda amostra, rf: pesos_markowitz(amostra, rf, aversao=5),
}
simulacoes_lambda = correr_backtest(estrategias_lambda, nome_corrida='Markowitz lambda 1 and 5')
mostrar_falhas('Markowitz lambda 1 and 5')

# Juntamos às simulações já feitas (λ=3, GMV e Max Sharpe) para comparar
simulacoes_comparar = {}
for (e, nome), sim in simulacoes.items():
    if nome in ['Markowitz (λ=3)', 'GMV', 'Max Sharpe', 'EW']:
        simulacoes_comparar[(e, nome)] = sim
simulacoes_comparar.update(simulacoes_lambda)

tabela_lambda = tabela_metricas(simulacoes_comparar, CUSTO)
tabela_lambda = tabela_lambda[tabela_lambda['Strategy'] != 'SPY (buy & hold)']
ordem_lambda = ['EW', 'GMV', 'Markowitz (λ=5)', 'Markowitz (λ=3)', 'Markowitz (λ=1)', 'Max Sharpe']
sensibilidade_lambda = tabela_lambda.groupby('Strategy')[['Ann. return', 'Ann. volatility', 'Sharpe', 'Max drawdown',
                                                          'Turnover (annual)', 'Effective N']].median().loc[ordem_lambda]
display(sensibilidade_lambda.round(3))

### 10.4 Os resultados dependem do período?

(a) Diferença de Sharpe face ao EW em cada experiência, ao longo do tempo (data de início da avaliação).
(b) Medianas por sub-período e por regime de mercado (ano de avaliação com o SPY a subir ou a descer).

In [ ]:
diferenca_ew = sharpe_largo.sub(sharpe_largo['EW'], axis=0)        # Sharpe da estratégia - Sharpe do EW
datas_inicio = pd.to_datetime(desenho['Evaluation start'])

estrategias_grafico = ['Markowitz (λ=3)', 'GMV', 'Max Sharpe', 'Risk parity (ERC)', 'Most diversified', 'Max decorrelation']
fig, axes = plt.subplots(2, 3, figsize=(17, 8), sharey=True)
for ax, nome in zip(axes.flatten(), estrategias_grafico):
    ax.scatter(datas_inicio, diferenca_ew[nome], color=cores[nome], s=20)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'{nome} minus EW (Sharpe)')
    ax.grid(True, linestyle='--', alpha=0.3)
for ax in axes[1]:
    ax.set_xlabel('Start of out-of-sample year')
plt.tight_layout()
plt.savefig('../graficos/04_diferenca_sharpe_vs_ew_no_tempo.png', dpi=150)
plt.show()

In [ ]:
def sub_periodo(data):
    # Agrupa o início do ano de avaliação em 3 sub-períodos
    if data.year < 2017:
        return '2012-2016'
    elif data.year < 2021:
        return '2017-2020'
    else:
        return '2021-2026'


# Regime de mercado: o SPY subiu ou desceu no ano de avaliação?
retorno_spy = resultados[resultados['Strategy'] == 'SPY (buy & hold)'].set_index('Experiment')['Ann. return']
regime = pd.Series(np.where(retorno_spy > 0, 'SPY up', 'SPY down'), index=retorno_spy.index)
periodo = datas_inicio.apply(sub_periodo)

resultados['Sub-period'] = resultados['Experiment'].map(periodo)
resultados['Market regime'] = resultados['Experiment'].map(regime)

print('Nº de experiências por sub-período:', periodo.value_counts().sort_index().to_dict())
print('Nº de experiências por regime:', regime.value_counts().to_dict())

sharpe_periodo = pd.pivot_table(resultados, index='Strategy', columns='Sub-period', values='Sharpe', aggfunc='median').loc[ordem]
sharpe_regime = pd.pivot_table(resultados, index='Strategy', columns='Market regime', values='Sharpe', aggfunc='median').loc[ordem]
retorno_regime = pd.pivot_table(resultados, index='Strategy', columns='Market regime', values='Ann. return', aggfunc='median').loc[ordem]

print('\nSharpe mediano por sub-período')
display(sharpe_periodo.round(3))
print('Sharpe mediano por regime de mercado')
display(sharpe_regime.round(3))
print('Retorno mediano por regime de mercado')
display(retorno_regime.round(3))

### 10.5 Os resultados dependem das ações sorteadas?

Peso médio dado a cada ação por cada estratégia (média dos rebalanceamentos, **só nas experiências em que a ação foi sorteada**).
Mostra para onde vai o dinheiro e se as estratégias dependem de poucas ações.

In [ ]:
soma_pesos = pd.DataFrame(0.0, index=acoes, columns=list(estrategias.keys()))
contagem_pesos = pd.DataFrame(0, index=acoes, columns=list(estrategias.keys()))

for (e, nome), sim in simulacoes.items():
    ativos_e = desenho.loc[e, 'Assets']
    for w in sim['pesos']:                      # um vetor de pesos por rebalanceamento
        for acao, peso in zip(ativos_e, w):
            soma_pesos.loc[acao, nome] += peso
            contagem_pesos.loc[acao, nome] += 1

peso_medio = soma_pesos / contagem_pesos

fig, ax = plt.subplots(figsize=(11, 10))
imagem = ax.imshow(peso_medio.values * 100, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(peso_medio.columns)))
ax.set_xticklabels(peso_medio.columns, rotation=35, ha='right')
ax.set_yticks(range(len(acoes)))
ax.set_yticklabels(acoes)
for i in range(len(acoes)):
    for j in range(len(peso_medio.columns)):
        valor = peso_medio.values[i, j] * 100
        ax.text(j, i, f'{valor:.0f}', ha='center', va='center', fontsize=8, color='white' if valor > 20 else 'black')
plt.colorbar(imagem, label='Average weight when selected (%)')
ax.set_title('Average weight of each stock when selected (%) - equal weight would be 8.3%')
plt.tight_layout()
plt.savefig('../graficos/04_peso_medio_por_acao.png', dpi=150)
plt.show()

### 10.6 Mesmos períodos, outras ações

Para ver se as conclusões dependem das ações sorteadas, repetimos o backtest com os **mesmos 100 períodos** mas:
- (a) um **novo sorteio de 12 ações** (outra semente). Como só há 23 ações, os dois sorteios têm em média cerca de metade das ações em comum;
- (b) **todas as ações elegíveis** (22 ou 23), ou seja, sem sorteio de ações.

Se o ranking das estratégias mudar muito entre as três versões, as conclusões dependem do subconjunto de ações.

In [ ]:
# (a) Mesmos períodos, OUTRAS 12 ações (outra semente, só para sortear as ações)
np.random.seed(SEMENTE + 1)
linhas = []
for e in desenho.index:
    inicio = desenho.loc[e, 'start_index']
    bloco = retornos[acoes].iloc[inicio:inicio + DIAS_DATASET]
    elegiveis = [acao for acao in acoes if bloco[acao].notna().all()]
    escolhidas = sorted(str(a) for a in np.random.choice(elegiveis, size=N_ATIVOS, replace=False))
    linhas.append({'Experiment': e, 'start_index': inicio, 'Assets': escolhidas})
desenho_outras = pd.DataFrame(linhas).set_index('Experiment')

em_comum = [len(set(a) & set(b)) for a, b in zip(desenho['Assets'], desenho_outras['Assets'])]
print('Nº médio de ações em comum com o sorteio principal:', round(np.mean(em_comum), 1), 'de', N_ATIVOS)

# (b) Mesmos períodos, TODAS as ações elegíveis
linhas = []
for e in desenho.index:
    inicio = desenho.loc[e, 'start_index']
    bloco = retornos[acoes].iloc[inicio:inicio + DIAS_DATASET]
    elegiveis = [acao for acao in acoes if bloco[acao].notna().all()]
    linhas.append({'Experiment': e, 'start_index': inicio, 'Assets': elegiveis})
desenho_todas = pd.DataFrame(linhas).set_index('Experiment')

simulacoes_outras = correr_backtest(estrategias, tabela_desenho=desenho_outras, nome_corrida='Other 12 stocks')
resultados_outras = tabela_metricas(simulacoes_outras, CUSTO, tabela_desenho=desenho_outras)
mostrar_falhas('Other 12 stocks')

simulacoes_todas = correr_backtest(estrategias, tabela_desenho=desenho_todas, nome_corrida='All eligible stocks')
resultados_todas = tabela_metricas(simulacoes_todas, CUSTO, tabela_desenho=desenho_todas)
mostrar_falhas('All eligible stocks')

In [ ]:
def resumo_curto(tabela):
    # Sharpe mediano, posição no ranking, retorno mediano e % de experiências que batem o EW
    sharpe_mediano = tabela.groupby('Strategy')['Sharpe'].median()
    return pd.DataFrame({
        'Median Sharpe': sharpe_mediano,
        'Rank': sharpe_mediano.rank(ascending=False),
        'Median return': tabela.groupby('Strategy')['Ann. return'].median(),
        '% beats EW (Sharpe)': percentagem_bate_ew(tabela)['% beats EW: Sharpe'],
    })


comparacao_acoes = pd.concat({
    '12 stocks (main)': resumo_curto(resultados),
    '12 stocks (new draw)': resumo_curto(resultados_outras),
    'All eligible stocks': resumo_curto(resultados_todas),
}, axis=1).loc[ordem]
display(comparacao_acoes.round(3))

# Concordância: em cada experiência, a estratégia bate (ou não) o EW nas DUAS amostras de 12 ações?
sharpe_principal = resultados.pivot(index='Experiment', columns='Strategy', values='Sharpe')
sharpe_outras = resultados_outras.pivot(index='Experiment', columns='Strategy', values='Sharpe')
bate_principal = sharpe_principal.gt(sharpe_principal['EW'], axis=0)
bate_outras = sharpe_outras.gt(sharpe_outras['EW'], axis=0)
concordancia = 100 * (bate_principal == bate_outras).mean()
print('% de experiências com a mesma conclusão "bate / não bate o EW" nas duas amostras de ações:')
display(concordancia.drop(index=['EW', 'SPY (buy & hold)']).round(0).to_frame('Agreement (%)'))

## 11. Guardar resultados (CSV + Excel)

- `04_desenho_experiencias.csv` — períodos e ações de cada experiência (permite reproduzir e reutilizar no projeto individual);
- `04_metricas_por_experiencia.csv` — todas as métricas, por experiência e estratégia;
- `04_falhas_otimizacao.csv` — todas as otimizações que falharam (nesses rebalanceamentos usámos pesos iguais); pode estar vazio;
- `04_estrategias_backtest.xlsx` — todas as tabelas de resumo, uma por folha (**usem este ficheiro para copiar tabelas para o relatório**).

In [ ]:
# Nota (Windows): feche os ficheiros .xlsx/.csv no Excel antes de correr esta célula,
# senão o Excel bloqueia-os e aparece um "PermissionError".

desenho_csv = desenho.copy()
desenho_csv['Assets'] = [', '.join(lista) for lista in desenho_csv['Assets']]
resultados.to_csv('../resultados/04_metricas_por_experiencia.csv', index=False, encoding='utf-8-sig')

# Registo de falhas: as que não têm 'Run' vieram da ilustração da secção 4
for falha in registo_falhas:
    if 'Run' not in falha:
        falha['Run'] = 'Illustration (section 4)'
tab_falhas = pd.DataFrame(registo_falhas, columns=['Run', 'Experiment', 'Rebalance date', 'Strategy', 'Message'])
print('Total de otimizações que falharam (todas as corridas):', len(tab_falhas))

tabelas = {
    'resumo_universo': resumo_universo,
    'pesos_ilustracao': pesos_ilustracao,
    'caracteristicas_ilustracao': caracteristicas,
    'iv_vs_erc': tab_rc,
    'mediana_metricas': mediana,
    'dispersao': dispersao,
    'bate_ew': bate_ew,
    'rankings': tab_rankings,
    'sharpe_por_custo': sharpe_por_custo,
    'ranking_por_custo': ranking_por_custo,
    'rebalance_semestral_vs_trim': comparacao_rebalance,
    'sensibilidade_lambda': sensibilidade_lambda,
    'sharpe_por_subperiodo': sharpe_periodo,
    'sharpe_por_regime': sharpe_regime,
    'retorno_por_regime': retorno_regime,
    'peso_medio_por_acao': peso_medio,
    'comparacao_acoes': comparacao_acoes,
    'desenho_experiencias': desenho_csv,
    'falhas_otimizacao': tab_falhas,
}

with pd.ExcelWriter('../resultados/04_estrategias_backtest.xlsx') as excel:
    for nome, tabela in tabelas.items():
        tabela.to_excel(excel, sheet_name=nome[:31])          # o Excel aceita no máximo 31 caracteres no nome da folha
        tabela.to_csv(f'../resultados/04_{nome}.csv', encoding='utf-8-sig')   # 'utf-8-sig' -> o Excel mostra bem λ e σ

print('Resultados guardados em ../resultados/')

## 12. Limitações (para a secção de robustez do relatório)

- **Viés de sobrevivência e de seleção:** as 23 ações foram escolhidas com base na capitalização **atual** (setembro de 2026).
  São empresas que sobreviveram e cresceram (NVDA, AVGO, LLY, …), o que **inflaciona os retornos absolutos de todas as estratégias**
  (inclusive o EW) face ao que um investidor teria obtido em tempo real. As comparações **entre** estratégias são menos afetadas,
  mas estratégias que concentram nos vencedores passados (Max Sharpe, Markowitz) podem beneficiar deste viés.
- **Experiências sobrepostas:** as 100 experiências partilham anos de avaliação → não são independentes; olhamos para medianas,
  dispersão e consistência e não para testes de significância.
- **Estimação:** médias históricas de 1 ano são muito ruidosas (erro de estimação) — afeta sobretudo Markowitz e Max Sharpe.
- **Custos simplificados:** custo proporcional fixo, sem impacto de mercado nem impostos; o SPY também tem uma comissão de gestão (~0.09%/ano) já incluída no preço.
- **Taxa sem risco:** a Treasury a 3 anos tem algum risco de taxa de juro; uma T-bill de 3 meses seria a alternativa mais "pura".
- **Universo:** apenas ações americanas de grande capitalização — os resultados podem não se aplicar a outras classes de ativos.

## 13. Resumo automático dos resultados

Esta célula escreve as principais conclusões **com os números desta execução** (assim o texto do relatório bate certo com as tabelas).

In [ ]:
estrategias_sem_spy = list(estrategias.keys())
med = mediana.loc[estrategias_sem_spy]          # medianas da secção 9.1, só as 8 estratégias

melhor_sharpe = med['Sharpe'].idxmax()
maior_retorno = med['Ann. return'].idxmax()
menor_vol = med['Ann. volatility'].idxmin()
menor_drawdown = med['Max drawdown'].idxmax()   # o drawdown é negativo: o "maior" é o menos mau
maior_turnover = med['Turnover (annual)'].idxmax()
mais_concentrada = med['Effective N'].idxmin()

print('PRINCIPAIS RESULTADOS (medianas das 100 experiências, líquidas de 10 bps)')
print(f"- Maior Sharpe: {melhor_sharpe} ({med.loc[melhor_sharpe, 'Sharpe']:.2f}) | EW: {med.loc['EW', 'Sharpe']:.2f} | SPY: {mediana.loc['SPY (buy & hold)', 'Sharpe']:.2f}")
print(f"- Maior retorno: {maior_retorno} ({med.loc[maior_retorno, 'Ann. return']:.1%}) | EW: {med.loc['EW', 'Ann. return']:.1%}")
print(f"- Menor volatilidade: {menor_vol} ({med.loc[menor_vol, 'Ann. volatility']:.1%}) | EW: {med.loc['EW', 'Ann. volatility']:.1%}")
print(f"- Menor drawdown máximo: {menor_drawdown} ({med.loc[menor_drawdown, 'Max drawdown']:.1%}) | EW: {med.loc['EW', 'Max drawdown']:.1%}")
print(f"- Maior turnover: {maior_turnover} ({med.loc[maior_turnover, 'Turnover (annual)']:.2f} por ano) | EW: {med.loc['EW', 'Turnover (annual)']:.2f}")
print(f"- Mais concentrada: {mais_concentrada} (N efetivo = {med.loc[mais_concentrada, 'Effective N']:.1f} de {N_ATIVOS})")

print('\n% DE EXPERIÊNCIAS EM QUE CADA ESTRATÉGIA BATE O EW (Sharpe | retorno | drawdown | volatilidade)')
for nome in bate_ew.index:
    print(f"- {nome:20s}: {bate_ew.loc[nome, '% beats EW: Sharpe']:4.0f}% | {bate_ew.loc[nome, '% beats EW: Ann. return']:4.0f}% | "
          f"{bate_ew.loc[nome, '% beats EW: Max drawdown']:4.0f}% | {bate_ew.loc[nome, '% beats EW: Ann. volatility']:4.0f}%")

print('\nROBUSTEZ')
mudaram = ranking_por_custo[ranking_por_custo.nunique(axis=1) > 1].index.tolist()
print(f'- Estratégias que mudam de posição no ranking entre 0 e 50 bps: {len(mudaram)} de {len(ranking_por_custo)} {mudaram}')
print(f"- Melhor estratégia (Sharpe mediano) com rebalanceamento trimestral: {comparacao_rebalance['Sharpe (quarterly)'].idxmax()}")
for versao in ['12 stocks (main)', '12 stocks (new draw)', 'All eligible stocks']:
    print(f"- Melhor estratégia com '{versao}': {comparacao_acoes[(versao, 'Median Sharpe')].idxmax()}")
print(f'- Otimizações que falharam (todas as corridas): {len(tab_falhas)}')

## 14. Conclusões e recomendação (guia para o relatório)

> Os números exatos são os do resumo automático (secção 13) e das tabelas acima. Os valores "≈" abaixo vêm de resultados que
> **não dependem da taxa sem risco** (retornos, volatilidade, drawdown, turnover, concentração). As afirmações sobre o **Sharpe**
> e sobre o **Max Sharpe** devem ser confirmadas com a vossa execução (com a série DGS3 real).

**1. O equally weighted é muito difícil de bater.** Nenhuma estratégia otimizada bate o EW em Sharpe de forma consistente:
as melhores (ERC, most diversified, inverse vol) ficam perto de 50% das experiências; o ERC e o inverse vol têm um Sharpe mediano
praticamente igual ao do EW e o most diversified fica ≈ 0.1 abaixo.
Com um novo sorteio de ações, ou com todas as ações (secção 10.6), a ordem entre estas estratégias muda → **as diferenças entre elas não são robustas**.

**2. Família de baixo risco (GMV, inverse vol, risk parity).** Reduzem a volatilidade face ao EW em ≈ 98–100% das experiências
e o drawdown máximo em ≈ 65–87%, mas ganham menos (retorno acima do EW em só ≈ 26–27% das experiências).
O inverse vol e o ERC fazem-no com turnover semelhante ao do EW (≈ 0.3 por ano) e carteiras bem diversificadas (N efetivo ≈ 11 de 12).
O GMV é mais concentrado (N efetivo ≈ 4) e tem o retorno mediano mais baixo.

**3. Estratégias que usam retornos esperados (Markowitz λ = 3 e Max Sharpe).** Carteiras muito concentradas
(Markowitz: N efetivo ≈ 1.5 e peso máximo médio ≈ 77%), turnover ≈ 8 vezes o do EW, mais volatilidade e drawdowns maiores.
O Markowitz tem o maior retorno mediano, mas também a maior dispersão (desvio-padrão do retorno anual ≈ 0.68 contra ≈ 0.22 no EW)
e só tem um drawdown menor do que o EW em ≈ 9% das experiências. É o efeito do **erro de estimação**: as médias de 1 ano
apontam para as ações que mais subiram, que nem sempre continuam a subir. Um λ maior (5) reduz o risco, mas não resolve a concentração.

**4. Most diversified e Max decorrelation** ficam a meio caminho: mais retorno do que a família de baixo risco,
mas turnover ≈ 3 vezes o do EW e concentração moderada (N efetivo ≈ 6–7).

**5. Custos de transação.** Com 10 bps o custo anual é pequeno para todas (mediana ≈ 0.3% ao ano no Markowitz, a estratégia mais cara;
≈ 1% na pior experiência) e o ranking pelo Sharpe **praticamente não muda** entre 0 e 50 bps: no máximo trocam de lugar estratégias
com Sharpe mediano quase igual, como o EW e o ERC (confirmar na secção 10.1 e no resumo da secção 13). Neste universo de ações muito líquidas, o que separa as estratégias é o erro de estimação, não os custos.

**6. Rebalanceamento trimestral** aumenta o turnover em ≈ 30–45% sem melhorar o desempenho de forma consistente →
o rebalanceamento **semestral** é suficiente.

**7. Dependência do período.** Nos anos em que o SPY caiu (12 experiências) o EW e as estratégias de baixo risco quase não perderam
(retorno mediano ≈ 0% a +4%), enquanto o Markowitz foi a única estratégia com retorno mediano negativo (≈ −5%, perto dos ≈ −5% do SPY).
O Markowitz também piora muito no sub-período 2021–2026. Os resultados variam muito entre experiências — por isso olhamos para medianas
e percentagens e não para uma única experiência.

**8. SPY.** O SPY tem o Sharpe mediano mais baixo e um dos retornos mais baixos (só o GMV rende menos), mas tem o **drawdown mediano
mais pequeno de todos** (≈ −9%) e uma volatilidade baixa (≈ 14%; só o GMV tem menos). A diferença de retorno reflete sobretudo o
**viés de seleção** do universo (as 23 ações são as maiores de **hoje**) e não uma vantagem real das estratégias sobre o mercado.

### Sugestão de recomendação (a decidir pelo grupo)

- **Investidor-alvo:** investidor de longo prazo com uma alocação a ações americanas que quer **reduzir a volatilidade e as quedas**
  com custos e complexidade baixos (por exemplo, um fundo de pensões ou um investidor particular prudente).
- **Recomendação:** **risk parity (ERC)** — ou o inverse vol, que é ainda mais simples. Têm um Sharpe semelhante ao EW,
  volatilidade mais baixa em praticamente todas as experiências, drawdowns menores na grande maioria, turnover baixo e pesos estáveis.
  O custo é um retorno menor do que o do EW (≈ 5 pontos percentuais por ano em mediana: ≈ 26% contra ≈ 31%).
- Para quem quer **mais retorno** e aceita a volatilidade das ações, o **EW** é a escolha mais simples e robusta (não precisa de estimar nada).
  O Markowitz e o Max decorrelation têm retorno mediano mais alto (≈ 40% e ≈ 33% contra ≈ 31%), mas com mais volatilidade, mais dispersão
  entre experiências e resultados que dependem das ações sorteadas (secção 10.6).
- **Não recomendamos** o Markowitz nem o Max Sharpe sem restrições: concentração extrema, muito turnover e resultados instáveis.
- **Quando reconsiderar:** (i) se as correlações entre ações mudarem muito (o ERC depende da matriz de covariância);
  (ii) em mercados muito concentrados em poucas ações vencedoras, onde o EW e o ERC ficam com pouco peso nessas ações;
  (iii) se os custos ou o universo forem muito diferentes (ex.: ações pouco líquidas); (iv) se forem adicionados limites de peso
  ou uma covariância *shrinkage* (extensões A e B do enunciado), que podem tornar o Markowitz e o Max Sharpe mais estáveis.